# YuvaIntern Week 4 – Supervised Learning Model Implementation

**Task:** Supervised Learning Model Implementation  
**Problem type:** Binary Classification  
**Dataset:** Breast Cancer Wisconsin (Diagnostic) Dataset  
**Model:** Logistic Regression

The goal is to predict the binary diagnosis target from numerical cell-measurement features. The target is used only for supervised learning and evaluation.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

df = pd.read_csv("YuvaIntern_Week4_Breast_Cancer_Dataset.csv")
features = list(df.columns[:30])

X = df[features]
y = df["target"]

print("Dataset shape:", df.shape)
print("Feature shape:", X.shape)
df.head()


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


## Preprocessing and Feature Engineering

All numerical predictors are standardized with StandardScaler. The scaler is placed inside a Scikit-learn Pipeline so that scaling parameters are learned only from the training data during validation. This avoids data leakage.


In [ ]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=5000, random_state=42))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)


In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1 Score :", round(f1, 4))


In [ ]:
print(classification_report(
    y_test, y_pred,
    target_names=["Malignant", "Benign"]
))


In [ ]:
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Malignant", "Benign"]
)
disp.plot()
plt.title("Confusion Matrix – Logistic Regression")
plt.show()


## Cross-Validation

Five-fold stratified cross-validation is used on the training data. Stratification keeps the class proportions reasonably consistent across folds.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    model, X_train, y_train,
    cv=cv,
    scoring="accuracy"
)

print("Fold accuracies:", np.round(cv_scores, 4))
print("Mean CV accuracy:", round(cv_scores.mean(), 4))
print("CV standard deviation:", round(cv_scores.std(), 4))


In [ ]:
plt.figure(figsize=(7,4))
plt.bar(range(1, 6), cv_scores)
plt.axhline(cv_scores.mean(), linestyle="--",
            label=f"Mean = {cv_scores.mean():.3f}")
plt.title("5-Fold Cross-Validation Accuracy")
plt.xlabel("Fold")
plt.ylabel("Accuracy")
plt.xticks(range(1, 6))
plt.legend()
plt.show()


## Strengths and Limitations

**Strengths:** Logistic Regression is relatively simple, interpretable, computationally efficient, and suitable for binary classification. The pipeline makes preprocessing reproducible and reduces the risk of data leakage.

**Limitations:** Logistic Regression assumes a linear relationship between the transformed predictors and the log-odds of the outcome. Complex nonlinear patterns may require models such as decision trees, random forests, or support vector machines. The dataset is also a specific diagnostic dataset, so results should not automatically be generalized to every population or clinical setting.


## Conclusion

The project implements a complete supervised-learning workflow: problem definition, data preparation, stratified train/test splitting, feature scaling, Logistic Regression training, test-set evaluation, confusion-matrix analysis, and five-fold cross-validation. The results provide a baseline classification model that can be improved or compared with other algorithms in future work.
